In [25]:
# HW0: robust mega acquisition (Colab-first, git-safe)
import os
import sys
import subprocess
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

REPO_URL = "https://github.com/egil10/stk-mat2011.git"
REPO_DIR = Path("/content/stk-mat2011") if IN_COLAB else Path.cwd().resolve().parents[1]

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        # keep notebook reproducible if you re-run later
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=False)

    # optional but useful when Colab VM is fresh
    req = REPO_DIR / "requirements.txt"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=False)

# make scripts importable
SCRIPTS_DIR = REPO_DIR / "code" / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

from p_duka import download_and_save

print(f"IN_COLAB={IN_COLAB}")
print(f"REPO_DIR={REPO_DIR}")
print(f"SCRIPTS_DIR={SCRIPTS_DIR}")

IN_COLAB=True
REPO_DIR=/content/stk-mat2011
SCRIPTS_DIR=/content/stk-mat2011/code/scripts


In [26]:
from datetime import datetime
import shutil
import pandas as pd

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

# Repo output path (where p_duka writes)
repo_processed = REPO_DIR / "code" / "data" / "processed"
repo_processed.mkdir(parents=True, exist_ok=True)

# Persistent cache on Drive (survives git pull/reclone)
drive_processed = Path("/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed") if IN_COLAB else repo_processed
if IN_COLAB:
    drive_processed.mkdir(parents=True, exist_ok=True)


def month_list(start_ym: str, end_ym: str):
    """Inclusive YYYYMM range."""
    y0, m0 = int(start_ym[:4]), int(start_ym[4:6])
    y1, m1 = int(end_ym[:4]), int(end_ym[4:6])
    out = []
    y, m = y0, m0
    while (y < y1) or (y == y1 and m <= m1):
        out.append(f"{y}{m:02d}")
        if m == 12:
            y, m = y + 1, 1
        else:
            m += 1
    return out


def expected_files(pair: str, yyyymm: str):
    s = pair.replace("/", "").lower()
    return [
        f"{s}_dukascopy_bid_{yyyymm}.parquet",
        f"{s}_dukascopy_ask_{yyyymm}.parquet",
    ]


def sync_drive_to_repo():
    if not IN_COLAB:
        return
    copied = 0
    for fp in drive_processed.glob("*.parquet"):
        tgt = repo_processed / fp.name
        if not tgt.exists():
            shutil.copy2(fp, tgt)
            copied += 1
    print(f"Synced Drive -> repo: {copied} new file(s)")


def sync_repo_to_drive():
    if not IN_COLAB:
        return
    copied = 0
    for fp in repo_processed.glob("*.parquet"):
        tgt = drive_processed / fp.name
        if not tgt.exists():
            shutil.copy2(fp, tgt)
            copied += 1
    print(f"Synced repo -> Drive: {copied} new file(s)")


print(f"repo_processed={repo_processed}")
print(f"drive_processed={drive_processed}")

Mounted at /content/drive
repo_processed=/content/stk-mat2011/code/data/processed
drive_processed=/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed


In [ ]:
# Fast acquisition: newest-first + chunked + periodic sync
PAIR = "EUR/USD"

# 1) Choose one chunk per run (recommended)
# Example quick chunk:
START_YYYYMM = "202001"
END_YYYYMM   = "202012"
SYNC_EVERY = 1
MAX_FAIL_BEFORE_STOP = 3

# Pull persistent files into repo first
sync_drive_to_repo()

months = month_list(START_YYYYMM, END_YYYYMM)
months = list(reversed(months))  # newest first
print(f"Planned months: {len(months)} ({months[-1]} -> {months[0]}) newest-first")

ok, skip, fail = [], [], []
ok_since_sync = 0

for i, yyyymm in enumerate(months, 1):
    names = expected_files(PAIR, yyyymm)
    have = all((drive_processed / n).exists() or (repo_processed / n).exists() for n in names)

    if have:
        skip.append(yyyymm)
        if i % 10 == 0:
            print(f"[{i}/{len(months)}] skip {yyyymm} (already exists)")
        continue

    print(f"[{i}/{len(months)}] fetch {PAIR} {yyyymm}")
    try:
        _ = download_and_save(PAIR, yyyymm, compression="snappy")
        ok.append(yyyymm)
        ok_since_sync += 1

        # periodic sync to persistent storage
        if ok_since_sync >= SYNC_EVERY:
            sync_repo_to_drive()
            ok_since_sync = 0

    except Exception as e:
        print(f"  ERROR {yyyymm}: {e}")
        fail.append(yyyymm)

        # safety stop if endpoint is flaky
        if len(fail) >= MAX_FAIL_BEFORE_STOP:
            print(f"Stopping early: reached {MAX_FAIL_BEFORE_STOP} failures.")
            break

# final sync
sync_repo_to_drive()

print("\nDone")
print(f"Downloaded: {len(ok)}")
print(f"Skipped existing: {len(skip)}")
print(f"Failed: {len(fail)}")
if ok:
    print("Newest downloaded months:", ok[:10])   # because newest-first
if fail:
    print("Failed months:", fail[:20])

Synced Drive -> repo: 0 new file(s)
Planned months: 12 (202101 -> 202112) newest-first
[1/12] fetch EUR/USD 202112


INFO:DUKASCRIPT:current timestamp :2021-12-01T09:35:42.141000
INFO:DUKASCRIPT:current timestamp :2021-12-01T14:59:14.782000
INFO:DUKASCRIPT:current timestamp :2021-12-02T00:54:56.288000
INFO:DUKASCRIPT:current timestamp :2021-12-02T12:54:52.003000
INFO:DUKASCRIPT:current timestamp :2021-12-02T18:35:48.185000
INFO:DUKASCRIPT:current timestamp :2021-12-03T11:16:33.645000
INFO:DUKASCRIPT:current timestamp :2021-12-03T15:00:02.668000
INFO:DUKASCRIPT:current timestamp :2021-12-03T21:55:15.631000
INFO:DUKASCRIPT:current timestamp :2021-12-06T12:42:48.567000
INFO:DUKASCRIPT:current timestamp :2021-12-07T01:43:13.218000
INFO:DUKASCRIPT:current timestamp :2021-12-07T13:41:14.081000
INFO:DUKASCRIPT:current timestamp :2021-12-08T06:09:25.181000
INFO:DUKASCRIPT:current timestamp :2021-12-08T13:30:59.164000
INFO:DUKASCRIPT:current timestamp :2021-12-08T23:54:21.063000
INFO:DUKASCRIPT:current timestamp :2021-12-09T14:23:32.065000
INFO:DUKASCRIPT:current timestamp :2021-12-10T07:20:43.686000
INFO:DUK

  Received 1,404,685 ticks
Synced repo -> Drive: 2 new file(s)
[2/12] fetch EUR/USD 202111


INFO:DUKASCRIPT:current timestamp :2021-11-01T13:23:58.159000
INFO:DUKASCRIPT:current timestamp :2021-11-02T05:31:55.091000
INFO:DUKASCRIPT:current timestamp :2021-11-02T15:04:23.992000
INFO:DUKASCRIPT:current timestamp :2021-11-03T11:54:59.670000
INFO:DUKASCRIPT:current timestamp :2021-11-03T18:24:37.695000
INFO:DUKASCRIPT:current timestamp :2021-11-04T04:46:23.358000
INFO:DUKASCRIPT:current timestamp :2021-11-04T13:38:49.275000
INFO:DUKASCRIPT:current timestamp :2021-11-05T01:10:11.189000
INFO:DUKASCRIPT:current timestamp :2021-11-05T12:36:35.951000
INFO:DUKASCRIPT:current timestamp :2021-11-05T20:02:00.450000
INFO:DUKASCRIPT:current timestamp :2021-11-08T12:51:19.761000
INFO:DUKASCRIPT:current timestamp :2021-11-09T05:17:38.022000
INFO:DUKASCRIPT:current timestamp :2021-11-09T15:15:01.138000
INFO:DUKASCRIPT:current timestamp :2021-11-10T08:57:19.170000
INFO:DUKASCRIPT:current timestamp :2021-11-10T15:13:20.867000
INFO:DUKASCRIPT:current timestamp :2021-11-11T00:00:06.163000
INFO:DUK

  Received 1,493,555 ticks
Synced repo -> Drive: 2 new file(s)
[3/12] fetch EUR/USD 202110


...

  Received 1,324,436 ticks
Synced repo -> Drive: 2 new file(s)
[9/12] fetch EUR/USD 202104


INFO:DUKASCRIPT:current timestamp :2021-04-01T12:14:03.328000
INFO:DUKASCRIPT:current timestamp :2021-04-02T01:48:08.610000
INFO:DUKASCRIPT:current timestamp :2021-04-02T20:37:05.298000
INFO:DUKASCRIPT:current timestamp :2021-04-05T15:00:13.292000
INFO:DUKASCRIPT:current timestamp :2021-04-06T09:09:51.358000
INFO:DUKASCRIPT:current timestamp :2021-04-06T19:00:38.155000
INFO:DUKASCRIPT:current timestamp :2021-04-07T09:48:40.942000
INFO:DUKASCRIPT:current timestamp :2021-04-07T18:38:18.115000
INFO:DUKASCRIPT:current timestamp :2021-04-08T11:59:00.736000
INFO:DUKASCRIPT:current timestamp :2021-04-09T01:26:08.571000
INFO:DUKASCRIPT:current timestamp :2021-04-09T13:03:44.861000
INFO:DUKASCRIPT:current timestamp :2021-04-12T03:56:19.369000
INFO:DUKASCRIPT:current timestamp :2021-04-12T15:00:24.137000
INFO:DUKASCRIPT:current timestamp :2021-04-13T08:19:38.926000
INFO:DUKASCRIPT:current timestamp :2021-04-13T14:42:09.332000
INFO:DUKASCRIPT:current timestamp :2021-04-14T07:17:43.172000
INFO:DUK

  Received 1,194,261 ticks
Synced repo -> Drive: 2 new file(s)
[10/12] fetch EUR/USD 202103


INFO:DUKASCRIPT:current timestamp :2021-03-01T08:50:01.055000
INFO:DUKASCRIPT:current timestamp :2021-03-01T14:51:51.176000
INFO:DUKASCRIPT:current timestamp :2021-03-02T00:39:46.494000
INFO:DUKASCRIPT:current timestamp :2021-03-02T09:01:34.745000
INFO:DUKASCRIPT:current timestamp :2021-03-02T14:49:17.804000
INFO:DUKASCRIPT:current timestamp :2021-03-02T23:59:09.917000
INFO:DUKASCRIPT:current timestamp :2021-03-03T10:05:07.438000
INFO:DUKASCRIPT:current timestamp :2021-03-03T14:44:32.319000
INFO:DUKASCRIPT:current timestamp :2021-03-03T19:53:56.722000
INFO:DUKASCRIPT:current timestamp :2021-03-04T07:47:34.417000
INFO:DUKASCRIPT:current timestamp :2021-03-04T13:22:46.041000
INFO:DUKASCRIPT:current timestamp :2021-03-04T17:27:27.242000
INFO:DUKASCRIPT:current timestamp :2021-03-04T19:52:23.219000
INFO:DUKASCRIPT:current timestamp :2021-03-05T05:06:25.319000
INFO:DUKASCRIPT:current timestamp :2021-03-05T10:36:56.796000
INFO:DUKASCRIPT:current timestamp :2021-03-05T14:19:08.324000
INFO:DUK

  Received 1,879,423 ticks
Synced repo -> Drive: 2 new file(s)
Synced repo -> Drive: 0 new file(s)

Done
Downloaded: 10
Skipped existing: 2
Failed: 0
Newest downloaded months: ['202112', '202111', '202110', '202109', '202108', '202107', '202106', '202105', '202104', '202103']


In [28]:
# Manifest / health check
store = drive_processed if IN_COLAB else repo_processed
files = sorted(store.glob("*.parquet"))
print(f"Store: {store}")
print(f"Total parquet files: {len(files)}")

total_mb = sum(f.stat().st_size for f in files) / (1024 * 1024)
print(f"Total size: {total_mb:.1f} MB")

rows = []
for fp in files:
    name = fp.name
    # expected format: eurusd_dukascopy_bid_202101.parquet
    parts = name.replace(".parquet", "").split("_")
    if len(parts) >= 4:
        yyyymm = parts[-1]
        side = parts[-2]
    else:
        yyyymm = ""
        side = ""
    rows.append({"file": name, "yyyymm": yyyymm, "side": side, "mb": fp.stat().st_size / 1e6})

m = pd.DataFrame(rows)
if len(m):
    print("\nFiles by month (head):")
    display(m.sort_values(["yyyymm", "side"]).head(20))

    month_counts = m.groupby("yyyymm").size().rename("n_files").reset_index().sort_values("yyyymm")
    print("\nMonth coverage (last 24):")
    display(month_counts.tail(24))

# Note for gitignore/git pull workflow:
# - Keep parquet in Drive path above (not in git).
# - Notebook syncs between repo output and Drive cache.
# - After git pull/reclone, data remains in Drive and is re-synced.

Store: /content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed
Total parquet files: 128
Total size: 2338.4 MB

Files by month (head):


,file,yyyymm,side,mb
0,eurusd_dukascopy_ask_202101.parquet,202101,ask,18.387604
64,eurusd_dukascopy_bid_202101.parquet,202101,bid,18.384328
1,eurusd_dukascopy_ask_202102.parquet,202102,ask,15.310751
65,eurusd_dukascopy_bid_202102.parquet,202102,bid,15.227004
2,eurusd_dukascopy_ask_202103.parquet,202103,ask,17.808417
66,eurusd_dukascopy_bid_202103.parquet,202103,bid,17.711489
3,eurusd_dukascopy_ask_202104.parquet,202104,ask,11.485313
67,eurusd_dukascopy_bid_202104.parquet,202104,bid,11.451401
4,eurusd_dukascopy_ask_202105.parquet,202105,ask,12.817079
68,eurusd_dukascopy_bid_202105.parquet,202105,bid,12.817309



Month coverage (last 24):


,yyyymm,n_files
40,202405,2
41,202406,2
42,202407,2
43,202408,2
44,202409,2
45,202410,2
46,202411,2
47,202412,2
48,202501,2
49,202502,2


In [30]:
# End-of-HW0 summary: Drive processed folder + month coverage
import re
from pathlib import Path

import pandas as pd

store = drive_processed if IN_COLAB else repo_processed
files = sorted(store.glob("*.parquet"))

# Filename pattern: eurusd_dukascopy_bid_202101.parquet
pat = re.compile(
    r"^(?P<sym>.+)_dukascopy_(?P<side>bid|ask)_(?P<ym>\d{6})\.parquet$",
    re.I,
)

rows = []
for fp in files:
    m = pat.match(fp.name)
    if not m:
        rows.append(
            {
                "file": fp.name,
                "symbol": "",
                "side": "",
                "yyyymm": "",
                "rows": None,
                "mb": fp.stat().st_size / 1e6,
            }
        )
        continue
    try:
        import pyarrow.parquet as pq
        nr = pq.ParquetFile(fp).metadata.num_rows
    except Exception:
        nr = len(pd.read_parquet(fp, columns=["datetime"]))

    rows.append(
        {
            "file": fp.name,
            "symbol": m.group("sym").upper(),
            "side": m.group("side").lower(),
            "yyyymm": m.group("ym"),
            "rows": int(nr),
            "mb": fp.stat().st_size / 1e6,
        }
    )

meta = pd.DataFrame(rows)
parsed = meta[meta["yyyymm"] != ""].copy()

total_bytes = sum(f.stat().st_size for f in files)
n_files = len(files)

print("=" * 60)
print("HW0 — processed store summary")
print("=" * 60)
print(f"Folder: {store}")
print(f"Parquet files: {n_files}")
print(f"Total size: {total_bytes / (1024**3):.3f} GB  ({total_bytes / (1024**2):.1f} MB)")

if len(parsed):
    bid = parsed[parsed["side"] == "bid"].rename(columns={"rows": "rows_bid", "mb": "mb_bid"})
    ask = parsed[parsed["side"] == "ask"].rename(columns={"rows": "rows_ask", "mb": "mb_ask"})
    cov = (
        bid[["symbol", "yyyymm", "rows_bid", "mb_bid", "file"]]
        .merge(
            ask[["symbol", "yyyymm", "rows_ask", "mb_ask", "file"]],
            on=["symbol", "yyyymm"],
            how="outer",
            suffixes=("_bidfile", "_askfile"),
        )
        .sort_values(["symbol", "yyyymm"])
    )
    cov["has_bid"] = cov["rows_bid"].notna()
    cov["has_ask"] = cov["rows_ask"].notna()
    cov["complete_pair"] = cov["has_bid"] & cov["has_ask"]
    cov["year"] = cov["yyyymm"].str[:4]

    rows_bid = int(cov["rows_bid"].fillna(0).sum())
    rows_ask = int(cov["rows_ask"].fillna(0).sum())

    print()
    print(f"Rows bid side (sum of bid files): {rows_bid:,}")
    print(f"Rows ask side (sum of ask files): {rows_ask:,}")
    print(f"Rows all files summed:            {rows_bid + rows_ask:,}")
    print()
    print(
        f"Months with complete bid+ask: {int(cov['complete_pair'].sum())} "
        f"/ {len(cov)} rows in coverage table"
    )
    print(f"Months missing bid: {int((~cov['has_bid']).sum())}")
    print(f"Months missing ask: {int((~cov['has_ask']).sum())}")

    print("\n--- By year (complete bid+ask months) ---")
    byy = (
        cov.groupby("year")
        .agg(
            months_total=("yyyymm", "nunique"),
            months_complete=("complete_pair", "sum"),
            rows_bid=("rows_bid", "sum"),
            rows_ask=("rows_ask", "sum"),
        )
        .sort_index()
    )
    display(byy)

    print("\n--- Month coverage (symbol, yyyymm) — tail ---")
    display(
        cov[
            [
                "symbol",
                "yyyymm",
                "has_bid",
                "has_ask",
                "complete_pair",
                "rows_bid",
                "rows_ask",
            ]
        ].tail(36)
    )

    # Optional: list gaps inside [min_ym, max_ym] for each symbol
    print("\n--- Missing YYYYMM inside observed range (per symbol) ---")
    for sym, g in cov.groupby("symbol"):
        yms = sorted(g["yyyymm"].unique())
        if not yms:
            continue
        lo, hi = yms[0], yms[-1]

        def nxt(ym):
            y, m = int(ym[:4]), int(ym[4:6])
            if m == 12:
                return f"{y+1}01"
            return f"{y}{m+1:02d}"

        have = set(yms)
        missing = []
        cur = lo
        while cur <= hi:
            if cur not in have:
                missing.append(cur)
            if cur == hi:
                break
            cur = nxt(cur)
        print(f"{sym}: range {lo}..{hi} | missing inside range: {len(missing)}")
        if missing:
            print("  ", missing[:40], ("..." if len(missing) > 40 else ""))

else:
    print("\n(No Dukascopy-style filenames parsed — check naming.)")

print()
print("Approx. unique tick times: use bid row sum when bid/ask are paired.")
print("=" * 60)

HW0 — processed store summary
Folder: /content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed
Parquet files: 128
Total size: 2.284 GB  (2338.4 MB)

Rows bid side (sum of bid files): 131,124,041
Rows ask side (sum of ask files): 131,124,041
Rows all files summed:            262,248,082

Months with complete bid+ask: 64 / 64 rows in coverage table
Months missing bid: 0
Months missing ask: 0

--- By year (complete bid+ask months) ---


,months_total,months_complete,rows_bid,rows_ask
year,,,,
2021,12,12,16797709,16797709
2022,12,12,36950672,36950672
2023,12,12,27553506,27553506
2024,12,12,20690554,20690554
2025,12,12,23463391,23463391
2026,4,4,5668209,5668209



--- Month coverage (symbol, yyyymm) — tail ---


,symbol,yyyymm,has_bid,has_ask,complete_pair,rows_bid,rows_ask
28,EURUSD,202305,True,True,True,2236715,2236715
29,EURUSD,202306,True,True,True,2051718,2051718
30,EURUSD,202307,True,True,True,2114645,2114645
31,EURUSD,202308,True,True,True,2238800,2238800
32,EURUSD,202309,True,True,True,1737379,1737379
33,EURUSD,202310,True,True,True,2199274,2199274
34,EURUSD,202311,True,True,True,1938713,1938713
35,EURUSD,202312,True,True,True,2113635,2113635
36,EURUSD,202401,True,True,True,2173014,2173014
37,EURUSD,202402,True,True,True,1709509,1709509



--- Missing YYYYMM inside observed range (per symbol) ---
EURUSD: range 202101..202604 | missing inside range: 0

Approx. unique tick times: use bid row sum when bid/ask are paired.
